# Séance 1 : Mission LaPosteImmobilier, modéliser la demande actuelle et en 2040

### Imports

In [ ]:
# À exécuter en premier sur Google Colab (sans effet en local) : récupère les données du cours
import os, sys
if "google.colab" in sys.modules:
    DEPOT = "https://github.com/borges67/export_cours_modelisation.git"
    DOSSIER = DEPOT.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.exists(f"/content/{DOSSIER}"):
        !git clone -q --depth 1 {DEPOT} /content/{DOSSIER}
    os.chdir(f"/content/{DOSSIER}")
print(os.getcwd())

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 120)
plt.rcParams["figure.figsize"] = (10, 3.5)

ROOT = Path.cwd()  # dossier du projet, qui contient data/

## 1. Découverte de la mission et des données transmises

In [ ]:
BATIMENTS = ["bureau_de_poste", "bureaux", "plateforme_tri", "plateforme_distribution"]
# toutes les feuilles de chaque fichier transmis par LPI, la première colonne en index (date_heure ou parametre)
LPI = {b: pd.read_excel(ROOT / "data" / "lpi" / f"lpi_actuel_{b}.xlsx", sheet_name=None, index_col=0) for b in BATIMENTS}

CONSO = {b: LPI[b]["consommation_2023"] for b in BATIMENTS}

In [ ]:
semaine = slice("2023-01-09", "2023-01-15")

fig, axes = plt.subplots(2, 2, figsize=(13, 6))
for ax, b in zip(axes.flat, BATIMENTS):
    c = CONSO[b].loc[semaine]
    ax.stackplot(c.index, c.T.values, labels=c.columns)
    ax.set_title(b)
    ax.set_ylabel("kW")
    ax.grid(alpha=0.3)
    ax.set_ylim(0, 360)
    ax.tick_params(axis="x", labelrotation=30, labelsize=7)
axes[0, 0].legend(fontsize=7, loc="upper left")
fig.suptitle("Consommation des bâtiments, une semaine de janvier 2023")
fig.tight_layout()
plt.show()

## 2. Décomposition de la demande : TS et nTS

In [ ]:
# Pour un type de bâtiment
# Construisez les trois courbes : nTS - TS chaud - TS froid
df_to_plot = pd.DataFrame({"nts": ..., "ts_joule": ..., "ts_froid": ...})


## 3. Demande de chaleur et année climatique

Une même quantité de chaleur ne demande pas la même quantité d'électricité selon l'équipement :
un radiateur Joule fournit 1 kWh de chaleur par kWhe, une PAC fournit COP kWh de chaleur par kWhe.
Pour comparer les équipements et pour changer d'année climatique, il faut donc revenir à la **demande de chaleur**.

De même, un climatiseur fournit EER kWh de froid par kWhe

In [ ]:
temp_df = pd.read_csv(ROOT / "data" / "interne" / "temperatures.csv", index_col="date_heure", parse_dates=True)  # une colonne par année climatique

df = pd.DataFrame({
    "chauffage_joule": conso["chauffage_joule"],
    "clim": conso["climatisation"],
    "temperature": temp_df["2023"],
})

# Demande de chaleur
df["dem_chaleur"] = ...

# Demande de froid
df["dem_froid"] = ...

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df.index, df["dem_chaleur"], color="tab:red", label="chaleur")
ax.plot(df.index, df["dem_froid"], color="tab:green", label="froid")
ax.set_ylabel("kW")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

Pour chacune des 168 heures de la semaine, on calcule un **coefficient normalisé** : des kW par
degré d'écart au seuil (sous 15 °C pour le chaud, au-dessus de 24 °C pour le froid). Il ne dépend
plus de la météo de 2023 : c'est une propriété du bâtiment (isolation, horaires de chauffage, taux
d'équipement).

In [ ]:
# Calculez l'écart au seuil de chauffage (15 °C) et de climatisation (24 °C), nul quand le seuil n'est pas franchi
df["ecart_chaud"] = ...  
df["ecart_froid"] = ...

# Heure de la semaine : 0 = lundi 0 h, ..., 167 = dimanche 23 h
df["heure_semaine"] = df.index.dayofweek * 24 + df.index.hour

# Calculez, pour chaque heure de la semaine, les kW par degré
# (une heure sans aucun écart donne 0 / 0 : remplacez le résultat par 0 avec .fillna(0))
df_group_week = df.groupby("heure_semaine")
coef_chaud = ...
coef_froid = ...

fig, ax = plt.subplots(figsize=(13, 4))
coef_chaud.plot(ax=ax, label="chaud (kW de chaleur par °C sous 15 °C)")
coef_froid.plot(ax=ax, label="froid (kW de froid par °C au-dessus de 24 °C)")
ax.set_xlabel("heure de la semaine (0 = lundi 0 h)")
ax.set_ylabel("kW par °C")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

In [ ]:
df["coef_chaud"] = df["heure_semaine"].map(coef_chaud)
df["coef_froid"] = df["heure_semaine"].map(coef_froid)

df = df.round(3)
df = df.rename(columns={"temperature":"T_2023", "dem_chaleur":"dem_chaleur_2023", "dem_froid":"dem_froid_2023"})
df["T_2012"] = temp_df["2012"]

# Construisez la demande de 2012
df["dem_chaleur_2012"] = ...
df["dem_froid_2012"] = ...

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df.index, df["dem_chaleur_2023"], color="tab:red", label="chaleur 2023")
ax.plot(df.index, df["dem_chaleur_2012"], color="tab:orange", label="chaleur 2012")
ax.plot(df.index, df["dem_froid_2023"], color="tab:blue", label="froid 2023")
ax.plot(df.index, df["dem_froid_2012"], color="tab:green", label="froid 2012")
ax.set_ylabel("kW")
ax.grid(alpha=0.3)
ax.legend()
ax.set_title("Demande de chaleur et de froid")
plt.show()

On repasse en électricité

In [ ]:
df = df.rename(columns={"chauffage_joule":"joule_2023", "clim":"clim_2023"})

df["dem_joule_2012"] = ...
df["dem_clim_2012"] = ...

### Modélisation PAC

Aujourd'hui, les bâtiments de LPI sont chauffés au Joule seulement. Prenons comme hypothèse que le bâtiment exemple passe complètement à la PAC.

La courbe unitaire de fonctionnement est dans `data/interne/courbe_pac.csv`.

Ces PAC couvrent tout le besoin jusqu'à −3 °C. En dessous, leur puissance baisse alors que le besoin
augmente : un **backup électrique** (rendement 1, comme un radiateur Joule) couvre le reste. Le
**COP effectif** est la chaleur fournie divisée par l'électricité consommée par l'ensemble PAC + backup.

COP = production chaleur / conso élec

In [ ]:
pac_df = pd.read_csv(ROOT / "data" / "interne" / "courbe_pac.csv")

# COP effectif de chaque heure --> le calculer dans Excel
df["cop_eff_pac"] = ...  # np.interp(températures, abscisses de la courbe, valeurs de la courbe)

# Électricité de la PAC = chaleur ÷ COP effectif
df["dem_pac"] = ...

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df.index, df["joule_2023"], color="tab:red", label="Joule")
ax.plot(df.index, df["dem_pac"], color="tab:orange", label="PAC")
ax.set_ylabel("kW")
ax.grid(alpha=0.3)
ax.legend()
ax.set_title("Demande électrique Joule vs PAC - 2023")
plt.show()

## 4. Flotte de véhicules électriques

Chaque fichier actuel donne, pour son bâtiment, la part de la flotte qui **arrive** et qui
**part** à chaque heure de l'année (onglets `ve_arrivees` et `ve_departs`). Le nombre de
véhicules branchés `N(t)` en découle :
`N(t) = N(0) + somme cumulée de N_total × (arrivées − départs)`.

In [ ]:
VE_ARRIVEES = LPI["bureaux"]["ve_arrivees"]["arrivees_part_flotte"]
VE_DEPARTS = LPI["bureaux"]["ve_departs"]["departs_part_flotte"]

df["ve_arrivees"] = VE_ARRIVEES
df["ve_departs"] = VE_DEPARTS

df

In [ ]:
semaine = slice("2023-01-09", "2023-01-15")
jour = slice("2023-01-09", "2023-01-09")

fig, ax = plt.subplots(figsize=(13, 4))
df["ve_arrivees"].loc[semaine].plot(ax=ax, label="arrivées")
df["ve_departs"].loc[semaine].plot(ax=ax, label="départs")
ax.set_xlabel("")
ax.set_ylabel("part de la flotte par heure")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

In [ ]:
# Construisez la part de la flotte branchée : somme cumulée des arrivées moins les départs
df["ve_stock"] = ...

fig, ax = plt.subplots(figsize=(13, 4))
df["ve_stock"].loc[semaine].plot(ax=ax, label="branchés")
ax.set_xlabel("")
ax.set_ylabel("part de la flotte branchée")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

### Consommation de référence (recharge non pilotée, donnée)

Chaque véhicule qui arrive charge à pleine puissance jusqu'à avoir récupéré l'énergie de sa
journée (km × kWh/km), pertes de charge comprises.

In [ ]:


# flux en véhicules par heure (les courbes sont en fraction de la flotte)
df["arrivees_flotte"] = ...
df["departs_flotte"] = ...

# Calculez l'énergie consommée par une tournée, l'énergie à tirer du réseau (pertes de charge comprises)
# et la durée de charge à la puissance de la borne
energie_trajet_kwh = ...                          # énergie consommée par la tournée
energie_reseau_kwh = ...                          # énergie à tirer du réseau
duree_charge_h = ...                              # heures à pleine puissance pour charger la batterie du véhicule


In [ ]:
# durée de charge arrondie à l'heure supérieure (un entier)
n_heures = ...

# véhicules en charge à l'heure t = véhicules arrivés durant les n_heures dernières heures
ve_en_charge = df["arrivees_flotte"].rolling(n_heures, min_periods=1).sum()

# Demande de recharge totale de la flotte
df["dem_ve"] = ...

fig, ax = plt.subplots(figsize=(13, 4))
df["dem_ve"].loc[jour].plot(ax=ax)
ax.set_xlabel("")
ax.set_ylabel("Demande de recharge (kW)")
ax.grid(alpha=0.3)
plt.show()

print(df["dem_ve"].loc[jour].sum().round(2))

## 5. Projection 2040 - Consommation bâtiment

#### Chaleur et froid

- Projections de LPI pour 2040 : `lpi_projections_2040.xlsx`

- demande chaleur normalisée 2040 = demande chaleur normalisée actuelle * facteur de rénovation

- années climatiques : 2023 (aujourd'hui), 2008, 2009 et 2012 ;
- chaleur (froid) de l'ensemble des bureaux = nombre de bâtiments × coefficient × écart au seuil ÷ taux d'équipement actuel (comme si tous étaient chauffés, climatisés), × facteur de rénovation pour la chaleur en 2040 ;
- demande électrique de l'ensemble des bureaux = nombre de bâtiments × nTS + taux Joule × chaleur + taux PAC × chaleur ÷ COP effectif + taux de climatisation × froid ÷ EER.

In [ ]:
PROJECTIONS = pd.read_excel(ROOT / "data" / "lpi" / "lpi_projections_2040.xlsx", sheet_name=None, index_col=0)
PARC = PROJECTIONS["parc_2040"]
PARC.T

In [ ]:
p = PARC.loc["bureaux"]
TAUX = pd.DataFrame({
    "actuel": [p["nombre_batiments_actuel"], p["taux_joule_actuel"], 0, p["taux_climatisation_actuel"], 1],  # pas de PAC aujourd'hui
    "2040": [p["nombre_batiments_2040"], p["taux_joule_2040"], p["taux_pac_2040"], p["taux_climatisation_2040"], p["facteur_renovation_chauffage_2040"]],
}, index=["nombre_batiments", "taux_joule", "taux_pac", "taux_climatisation", "renovation"])
TAUX

In [ ]:
HORIZONS = ["actuel", "2040"]
ANNEES = ["2008", "2009", "2012"]

# Pour chaque année climatique : 
# 1. Récupérer la demande en chaleur / froid d'un bâtiment
# 2. Calculer la demande en chaleur / froid totale de tous les bâtiments (du type) pour actuel / 2040
# 3. Calculer demande électrique totale de tous les bâtiments (du type) pour actuel / 2040

In [ ]:
semaine_froide = slice("2023-02-06", "2023-02-12")  # vague de froid de février 2012

df = ... # consommation calculée précédemment

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, h in zip(axes, HORIZONS):
    d = df.loc[semaine_froide, [f"nts_{h}", f"joule_{h}_2012", f"pac_{h}_2012", f"clim_{h}_2012"]]
    ax.stackplot(d.index, d.T.values, labels=["nts", "joule", "pac", "clim"])
    ax.set_title(f"Ensemble des bureaux, {h}, semaine froide (climat 2012)")
    ax.grid(alpha=0.3)
    ax.tick_params(axis="x", labelrotation=30, labelsize=7)
axes[0].set_ylabel("kW")
axes[0].legend(fontsize=7, loc="upper left")
fig.tight_layout()
plt.show()

#### Véhicules électriques

In [ ]:
# 1. Récupérer demande VE actuelle
# 2. Rescale pour atteindre demande VE 2040
# 3. L'ajouter au total

FLOTTE_2040 = ...

# Construisez la recharge de l'ensemble des bureaux, aujourd'hui et en 2040 : même recharge par véhicule
# qu'aujourd'hui (mêmes km et même borne en 2040), flotte par bâtiment de FLOTTE_2040 en 2040
df["ve_actuel"] = ...
df["ve_2040"] = ...
(df[["ve_actuel", "ve_2040"]].sum() / 1000).round(0)  # MWh/an

#### Demande totale des bureaux

In [ ]:
for h in HORIZONS:
    for a in ANNEES:
        df[f"total_{h}_{a}"] = ...  # bâtiments + véhicules

pd.DataFrame({(h, a): {"énergie (GWh/an)": df[f"total_{h}_{a}"].sum() / 1e6, "pointe (MW)": df[f"total_{h}_{a}"].max() / 1000}
              for h in HORIZONS for a in ANNEES}).T.round(2)

#### Agrégation

In [ ]:
taux_co2 = pd.read_csv(ROOT / "data" / "interne" / "courbe_co2.csv", index_col="date_heure", parse_dates=True)["co2_g_par_kwh"]  # contenu CO2 horaire éCO2mix 2023 (g/kWh)

# Complétez les émissions (tCO2/an) : somme sur l'année de la demande (kW) × contenu CO2 de l'heure (g/kWh)
RESTITUTION = pd.DataFrame({(h, a): {"energie_gwh": df[f"total_{h}_{a}"].sum() / 1e6,
                                     "pointe_mw": df[f"total_{h}_{a}"].max() / 1000,
                                     "emissions_tco2": ...,
                                     }
                            for h in HORIZONS for a in ANNEES}).T
RESTITUTION.round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, (col, titre) in zip(axes, [("energie_gwh", "Énergie électrique (GWh/an)"), ("pointe_mw", "Pointe électrique (MW)"),
                                   ("emissions_tco2", "Émissions, contenu moyen 2023 (tCO2/an)")]):
    RESTITUTION[col].unstack(0).loc[ANNEES, HORIZONS].plot.bar(ax=ax, rot=0, title=titre)
    ax.set_xlabel("année climatique")
    ax.grid(alpha=0.3)
fig.suptitle("Ensemble des bureaux de LPI : ce qu'on présenterait au client")
fig.tight_layout()
plt.show()